# Medical Image Segmentation with U-Net (Kvasir-SEG Dataset)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
[![PyTorch](https://img.shields.io/badge/PyTorch-EE4C2C?style=flat&logo=pytorch&logoColor=white)](https://pytorch.org/)
[![Dataset](https://img.shields.io/badge/Dataset-Kvasir--SEG-orange?style=flat)](https://datasets.simula.no/kvasir-seg/)

## Overview
This Google Colab notebook implements an end-to-end deep learning pipeline for **gastrointestinal polyp segmentation** from colonoscopy images using the **U-Net** architecture on the **Kvasir-SEG** dataset.

### Key Workflow Steps:
1. **Environment Setup and GPU Check**: Verify hardware acceleration (CUDA / T4 GPU).
2. **Dataset Downloading and Preprocessing**: Automated download, extraction, resizing, and normalization.
3. **PyTorch Custom Dataset**: Dataset loader and ground-truth mask binarization.
4. **U-Net Architecture**: Implementation of encoder, bottleneck, decoder, and skip connections.
5. **Evaluation Metrics**: Dice Similarity Coefficient (DSC) and Intersection over Union (IoU / Jaccard Index).
6. **Training and Evaluation Pipeline**: Optimization with Binary Cross-Entropy (BCE) loss and Adam optimizer.
7. **Qualitative Visualization**: Side-by-side comparison of Input Images, Ground Truth Masks, and Predicted Masks.
8. **Model Checkpoint Export**: Saving and downloading trained weights.

### Hardware Accelerator Check
> **Note for Google Colab Users:** Make sure to enable GPU acceleration for faster training.
> Go to **Runtime** > **Change runtime type** > Select **T4 GPU** (or any available GPU).

In [ ]:
# Check PyTorch and GPU Availability
import torch

print(f"PyTorch Version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Compute Device: {device}")

if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
else:
    print("Running on CPU. (For faster training, enable GPU in Colab Runtime settings)")

## Step 0: Imports and Global Configuration
Import all necessary libraries and configure the training hyperparameters.

In [ ]:
import os
import ssl
import urllib.request
import zipfile
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Hyperparameters and Configurations
DATASET_URL = 'https://datasets.simula.no/downloads/kvasir-seg.zip'
DATASET_ZIP = 'kvasir-seg.zip'
DATASET_DIR = 'Kvasir-SEG'

CONFIG = {
    'img_size': 128,        # Image resize dimensions (128x128)
    'batch_size': 8,        # Batch size for DataLoader
    'epochs': 5,            # Number of training epochs (increase for better accuracy)
    'lr': 1e-3,             # Adam optimizer learning rate
    'subset_size': 100,     # Number of images to use (set 0 or None to use all 1000 images)
    'save_model_path': 'unet_kvasir_seg.pth',
    'output_plot_path': 'segmentation_results.png'
}

print("Configurations loaded:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## Step 1: Dataset Downloading and Preprocessing
The **Kvasir-SEG** dataset contains 1,000 polyp images along with their corresponding ground truth segmentation masks.
The function below downloads the zip archive and extracts the image and mask files.

In [ ]:
def download_and_extract_dataset(dataset_dir=DATASET_DIR, dataset_url=DATASET_URL, dataset_zip=DATASET_ZIP):
    """
    Downloads and extracts the Kvasir-SEG dataset if not already present.
    """
    if not os.path.exists(dataset_dir):
        print(f"Downloading Kvasir-SEG dataset from {dataset_url}...")
        # Bypass SSL certificate verification if needed in restricted environments
        ssl._create_default_https_context = ssl._create_unverified_context

        if not os.path.exists(dataset_zip):
            urllib.request.urlretrieve(dataset_url, dataset_zip)
            print(f"Downloaded {dataset_zip} successfully.")

        print("Extracting dataset archive...")
        with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
            zip_ref.extractall()
        print(f"Dataset successfully extracted to '{dataset_dir}'.")
    else:
        print(f"Dataset directory '{dataset_dir}' is already present.")

# Download and extract
download_and_extract_dataset()

## Step 2: PyTorch Custom Dataset and Data Loaders

We define `MedicalImageDataset` which loads paired images and binary masks:
- **Images**: Resized to (128, 128), converted to Tensor, and normalized using ImageNet mean and std.
- **Masks**: Grayscale mode (`L`), resized, converted to Tensor, and binarized with threshold > 0.5 yielding float values in {0.0, 1.0}.

In [ ]:
class MedicalImageDataset(Dataset):
    """
    Custom PyTorch Dataset for loading medical images and corresponding binary masks.
    """
    def __init__(self, image_dir, mask_dir, img_size=(128, 128), transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size
        self.transform = transform

        # Collect sorted list of image file names
        self.images = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])

        self.mask_transform = transforms.Compose([
            transforms.Resize(self.img_size),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.transform:
            image = self.transform(image)
        else:
            default_img_transform = transforms.Compose([
                transforms.Resize(self.img_size),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
            image = default_img_transform(image)

        mask = self.mask_transform(mask)
        # Binarize mask to 0.0 or 1.0
        mask = (mask > 0.5).float()

        return image, mask

# Setup Image Transformations
transform = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

img_dir = os.path.join(DATASET_DIR, 'images')
mask_dir = os.path.join(DATASET_DIR, 'masks')

full_dataset = MedicalImageDataset(img_dir, mask_dir, img_size=(CONFIG['img_size'], CONFIG['img_size']), transform=transform)
print(f"Total images available in dataset: {len(full_dataset)}")

# Create dataset subset (or use full dataset)
if CONFIG['subset_size'] and CONFIG['subset_size'] < len(full_dataset):
    dataset = Subset(full_dataset, range(CONFIG['subset_size']))
    print(f"Using subset of {CONFIG['subset_size']} samples for fast experimentation.")
else:
    dataset = full_dataset
    print(f"Using full dataset of {len(full_dataset)} samples.")

train_loader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True)

# Inspect a sample batch
sample_imgs, sample_masks = next(iter(train_loader))
print(f"Sample Batch Images Shape: {sample_imgs.shape} (Batch, Channels, Height, Width)")
print(f"Sample Batch Masks Shape:  {sample_masks.shape} (Batch, Channels, Height, Width)")

### Exploratory Data Visualization
Let's visualize a few input colonoscopy images alongside their expert ground-truth polyp masks.

In [ ]:
def display_sample_pairs(dataset, num_samples=3):
    inv_normalize = transforms.Normalize(
        mean=[-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.225],
        std=[1 / 0.229, 1 / 0.224, 1 / 0.225]
    )
    
    fig, axs = plt.subplots(num_samples, 2, figsize=(8, 3.5 * num_samples))
    for i in range(num_samples):
        img, mask = dataset[i]
        img_denorm = inv_normalize(img).permute(1, 2, 0).numpy()
        img_denorm = np.clip(img_denorm, 0, 1)
        
        axs[i, 0].imshow(img_denorm)
        axs[i, 0].set_title(f"Sample #{i+1}: Endoscopy Image", fontweight='bold')
        axs[i, 0].axis('off')
        
        axs[i, 1].imshow(mask.squeeze().numpy(), cmap='gray')
        axs[i, 1].set_title(f"Sample #{i+1}: Ground Truth Mask", fontweight='bold')
        axs[i, 1].axis('off')
        
    plt.tight_layout()
    plt.show()

display_sample_pairs(full_dataset, num_samples=3)

## Step 3: U-Net Architecture

The **U-Net** architecture consists of:
1. **Contracting Path (Encoder)**: Captures multi-scale contextual features through repeated `DoubleConv` (Conv3x3 -> BatchNorm -> ReLU x2) and 2x2 Max Pooling.
2. **Bottleneck**: Highest abstraction feature representation.
3. **Expanding Path (Decoder)**: Upsampling via Transposed Convolutions (`ConvTranspose2d`) concatenated with corresponding encoder skip connections to restore fine spatial localization.
4. **Final Output Layer**: 1x1 Convolution + Sigmoid activation yielding probability map in [0, 1].

In [ ]:
class DoubleConv(nn.Module):
    """(Convolution => [BatchNorm] => ReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    """
    U-Net architecture for 2D Medical Image Segmentation.
    """
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Encoder (Contracting Path)
        self.down1 = DoubleConv(in_channels, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)

        # Decoder (Expanding Path)
        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(128, 64)

        # Output Layer
        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.down1(x)             # 64 channels
        x2 = self.pool(x1)
        x3 = self.down2(x2)            # 128 channels
        x4 = self.pool(x3)
        x5 = self.down3(x4)            # 256 channels (bottleneck)

        # Decoder with Skip Connections
        x = self.up1(x5)               # 128 channels
        x = torch.cat([x, x3], dim=1)  # 128 + 128 = 256 channels
        x = self.conv1(x)              # 128 channels

        x = self.up2(x)                # 64 channels
        x = torch.cat([x, x1], dim=1)  # 64 + 64 = 128 channels
        x = self.conv2(x)              # 64 channels

        return torch.sigmoid(self.out_conv(x))

# Instantiate model and verify parameter counts
model = UNet(in_channels=3, out_channels=1).to(device)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"U-Net Model initialized on device: {device}")
print(f"Total Trainable Parameters: {num_params:,}")

# Test dummy forward pass
dummy_input = torch.randn(2, 3, CONFIG['img_size'], CONFIG['img_size']).to(device)
with torch.no_grad():
    dummy_output = model(dummy_input)
print(f"Forward pass check - Input shape: {dummy_input.shape} -> Output shape: {dummy_output.shape}")

## Step 4: Evaluation Metrics and Loss Function

### Mathematical Formulations:
- **Binary Cross-Entropy (BCE) Loss**:
  $$\mathcal{L}_{\text{BCE}} = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

- **Dice Similarity Coefficient (DSC)**:
  $$\text{Dice} = \frac{2 \cdot |Y \cap \hat{Y}| + \epsilon}{|Y| + |\hat{Y}| + \epsilon}$$

- **Intersection over Union (IoU / Jaccard Index)**:
  $$\text{IoU} = \frac{|Y \cap \hat{Y}| + \epsilon}{|Y \cup \hat{Y}| + \epsilon}$$

In [ ]:
def dice_coeff(pred, target, smooth=1.0, threshold=0.5):
    """Computes Dice Similarity Coefficient (DSC)."""
    if threshold is not None:
        pred = (pred > threshold).float()

    pred_flat = pred.contiguous().view(-1)
    target_flat = target.contiguous().view(-1)

    intersection = (pred_flat * target_flat).sum()
    dice = (2.0 * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)
    return dice


def iou_score(pred, target, smooth=1.0, threshold=0.5):
    """Computes Intersection over Union (IoU) / Jaccard Index."""
    if threshold is not None:
        pred = (pred > threshold).float()

    pred_flat = pred.contiguous().view(-1)
    target_flat = target.contiguous().view(-1)

    intersection = (pred_flat * target_flat).sum()
    total = pred_flat.sum() + target_flat.sum()
    union = total - intersection

    iou = (intersection + smooth) / (union + smooth)
    return iou

# Loss function and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])
print(f"Loss Function: {criterion}")
print(f"Optimizer: {optimizer}")

## Step 5: Training and Evaluation Loop
We train the model over the specified number of epochs while logging the **BCE Loss**, **Dice Coefficient**, and **IoU Score** per epoch.

In [ ]:
history = {
    'loss': [],
    'dice': [],
    'iou': []
}

print("=" * 70)
print(f"Starting U-Net Training for {CONFIG['epochs']} Epochs on {device}...")
print("=" * 70)

start_time = time.time()

for epoch in range(CONFIG['epochs']):
    model.train()
    epoch_loss = 0.0
    epoch_dice = 0.0
    epoch_iou = 0.0
    
    epoch_start = time.time()

    for batch_idx, (images, masks) in enumerate(train_loader):
        images = images.to(device)
        masks = masks.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        # Backward pass & optimization
        loss.backward()
        optimizer.step()

        # Accumulate metrics
        epoch_loss += loss.item()
        epoch_dice += dice_coeff(outputs, masks).item()
        epoch_iou += iou_score(outputs, masks).item()

    num_batches = len(train_loader)
    avg_loss = epoch_loss / num_batches
    avg_dice = epoch_dice / num_batches
    avg_iou = epoch_iou / num_batches

    history['loss'].append(avg_loss)
    history['dice'].append(avg_dice)
    history['iou'].append(avg_iou)

    epoch_duration = time.time() - epoch_start
    print(f"Epoch [{epoch+1:02d}/{CONFIG['epochs']:02d}] ({epoch_duration:.1f}s) | Loss: {avg_loss:.4f} | Dice: {avg_dice:.4f} | IoU: {avg_iou:.4f}")

total_time = time.time() - start_time
print("=" * 70)
print(f"Training finished successfully in {total_time:.2f} seconds!")

### Step 5b: Plotting Training Curves

In [ ]:
epochs_range = range(1, CONFIG['epochs'] + 1)

plt.figure(figsize=(14, 4))

# Loss Plot
plt.subplot(1, 3, 1)
plt.plot(epochs_range, history['loss'], 'r-o', label='BCE Loss')
plt.title('Training Loss', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

# Dice Plot
plt.subplot(1, 3, 2)
plt.plot(epochs_range, history['dice'], 'g-o', label='Dice Score')
plt.title('Dice Similarity Coefficient', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Dice')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

# IoU Plot
plt.subplot(1, 3, 3)
plt.plot(epochs_range, history['iou'], 'b-o', label='IoU Score')
plt.title('Intersection over Union (IoU)', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('IoU')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.show()

## Step 6: Qualitative Visualization

Let's visualize the model predictions in comparison with the ground-truth polyp masks.
Each row displays:
1. **Input RGB Image** (denormalized)
2. **Ground Truth Mask** (expert manual annotation)
3. **Predicted Mask** (output from U-Net with threshold > 0.5)

In [ ]:
def visualize_predictions(model, dataloader, device, save_path=CONFIG['output_plot_path'], num_samples=3):
    """
    Plots and saves comparison of Input Image, Ground Truth Mask, and Predicted Mask.
    """
    model.eval()
    images, masks = next(iter(dataloader))

    images = images.to(device)
    masks = masks.to(device)

    with torch.no_grad():
        outputs = model(images)

    images = images.cpu()
    masks = masks.cpu()
    outputs = outputs.cpu()

    inv_normalize = transforms.Normalize(
        mean=[-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.225],
        std=[1 / 0.229, 1 / 0.224, 1 / 0.225]
    )

    num_samples = min(num_samples, images.size(0))
    fig, axs = plt.subplots(num_samples, 3, figsize=(11, 3.8 * num_samples))

    if num_samples == 1:
        axs = np.expand_dims(axs, axis=0)

    for i in range(num_samples):
        img = inv_normalize(images[i]).permute(1, 2, 0).numpy()
        img = np.clip(img, 0, 1)
        
        gt_mask = masks[i].squeeze().numpy()
        pred_mask = (outputs[i].squeeze() > 0.5).numpy().astype(np.float32)

        # Input Image
        axs[i, 0].imshow(img)
        axs[i, 0].set_title(f"Sample {i+1}: Input Image", fontsize=12, fontweight='bold')
        axs[i, 0].axis("off")

        # Ground Truth Mask
        axs[i, 1].imshow(gt_mask, cmap='gray')
        axs[i, 1].set_title(f"Sample {i+1}: Ground Truth Mask", fontsize=12, fontweight='bold')
        axs[i, 1].axis("off")

        # Predicted Mask
        axs[i, 2].imshow(pred_mask, cmap='gray')
        axs[i, 2].set_title(f"Sample {i+1}: Predicted Mask (U-Net)", fontsize=12, fontweight='bold')
        axs[i, 2].axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Qualitative comparison saved to '{save_path}'.")
    plt.show()

# Generate visualization
visualize_predictions(model, train_loader, device, num_samples=3)

## Step 7: Save Weights and Colab File Export
Save model weights to a `.pth` file.

In [ ]:
# Save model checkpoint
torch.save(model.state_dict(), CONFIG['save_model_path'])
print(f"Model weights saved to '{CONFIG['save_model_path']}'")

# Colab download helper (optional)
try:
    from google.colab import files
    # Uncomment below to automatically download to local machine from Colab:
    # files.download(CONFIG['save_model_path'])
    # files.download(CONFIG['output_plot_path'])
    print("Google Colab detected. Use files.download(...) if you wish to export files.")
except ImportError:
    print("Running in standard Python environment.")